# Steer2Edit native reproduction — LLaMA-2-7B-Chat safety alignment

Target: **average safety (refusal rate) = 88.15**, components
GCG 95.1 / ADV-LLM 81.2, from arXiv 2602.09870v4 (8 Nov 2026, NeurIPS 2026) Table 5,
cross-checked against Table 11's 88.1. Official code pinned at `86382c2`.

Contract: `docs/STEER2EDIT_REPRODUCTION_CONTRACT.md`. Study and discrepancy map (D1-D10):
`docs/STEER2EDIT_DEEP_STUDY_AND_CONTRACT.md`.

## What this notebook does and does not do

`PREFLIGHT` is CPU-only and needs no approval: it detects the platform, checks dependencies,
pins the repository, validates the released data, and runs the edit-math suite. It reports a
concrete resource estimate and then **stops**.

Every stage that downloads weights or generates text refuses to run until `OWNER_APPROVED = True`.
A green preflight is **not** a reproduction. The final cell prints the run state explicitly so
setup can never be read as completed generation.

Runs on Google Colab, Kaggle or a generic cloud host (e.g. Nebius). Note the precision issue: the
released code hardcodes bfloat16, which **T4 and P100 do not support**. On those GPUs this
notebook uses float16 and records the deviation.

**Scope.** This is the *text-LLM* native reproduction. Steer2Edit's safety experiments are text
jailbreaks; its vision-language appendix edits the LLM backbone but still evaluates text-only
benchmarks. The MM-SafetyBench four-condition evaluation (TEXT / SD / TYPO / SD_TYPO) is a
separate adaptation notebook, written only after this native check returns a number.

## 1. Setup: run state, platform detection and resource estimate

In [ ]:
# --- Run state -----------------------------------------------------------------
# PREFLIGHT : CPU only, no weights, no generation. Safe to run now.
# SMOKE     : 8 prompts, 1 config. Proves the pipeline end to end.
# PILOT     : 100 prompts, published config only (contract arm (a)).
# FULL      : 100 prompts, 125-config grid (contract arm (b)).
STAGE = "PREFLIGHT"

# Gate for anything that downloads weights or generates text.
OWNER_APPROVED = False

PUBLISHED = {'avg_safety': 88.15, 'gcg': 95.1, 'advllm': 81.2, 'avg_utility': 28.0}
BEST_CFG = {'rho_attn': 0.18, 'rho_mlp': 0.55, 'alpha': 0.8}
REPO_COMMIT = "86382c2f44fdfda2bafe80faf09634e6d729562c"
REPO_URL = "https://github.com/Trustworthy-ML-Lab/Steer2Edit.git"
BASE_MODEL = "meta-llama/Llama-2-7b-chat-hf"
# D10: the paper and code pin no revision. Set this before any scientific run and record it.
BASE_MODEL_REVISION = None

import json, os, platform, shutil, subprocess, sys
from pathlib import Path

def detect_platform() -> dict:
    env = {"host": "generic", "scratch": Path("/tmp"), "work": Path.cwd()}
    if "google.colab" in sys.modules or os.path.isdir("/content"):
        env.update(host="colab", scratch=Path("/content"), work=Path("/content"))
    elif os.path.isdir("/kaggle"):
        # /kaggle/working is quota-limited; /kaggle/temp is the larger scratch area.
        env.update(host="kaggle", scratch=Path("/kaggle/temp"), work=Path("/kaggle/working"))
    return env

ENV = detect_platform()

def gpu_report() -> dict:
    try:
        import torch
    except ImportError:
        return {"torch": None, "cuda": False, "gpus": []}
    info = {"torch": torch.__version__, "cuda": torch.cuda.is_available(), "gpus": []}
    for i in range(torch.cuda.device_count() if info["cuda"] else 0):
        p = torch.cuda.get_device_properties(i)
        info["gpus"].append({
            "name": p.name, "total_gib": round(p.total_memory / 2**30, 1),
            "capability": f"sm_{p.major * 10 + p.minor}", "bf16": p.major >= 8,
        })
    return info

GPU = gpu_report()
N_GPU = len(GPU["gpus"])
BF16_OK = bool(N_GPU) and all(g["bf16"] for g in GPU["gpus"])

# Released code hardcodes bfloat16 (D7). Honour it only where the hardware can.
DTYPE = "bfloat16" if BF16_OK else "float16"
TENSOR_PARALLEL_SIZE = max(1, N_GPU)   # released default is 4; we use what exists

DEVIATIONS = []
if N_GPU and not BF16_OK:
    DEVIATIONS.append(
        "precision: released pipeline uses bfloat16; this hardware lacks sm_80+ so float16 is "
        "used instead (fp16 has 10 mantissa bits vs bf16's 8, but a far smaller exponent range; "
        f"relevant because |g| is compared against a gate of rho*alpha = "
        f"{BEST_CFG['rho_attn'] * BEST_CFG['alpha']:.3f})"
    )
if N_GPU and TENSOR_PARALLEL_SIZE != 4:
    DEVIATIONS.append(
        f"tensor_parallel_size={TENSOR_PARALLEL_SIZE} rather than the released default of 4"
    )
if BASE_MODEL_REVISION is None:
    DEVIATIONS.append(
        "checkpoint revision not pinned (D10) - set BASE_MODEL_REVISION before a scientific run"
    )

free_gib = round(shutil.disk_usage(ENV["scratch"]).free / 2**30, 1)

print(f"stage              : {STAGE}   owner_approved={OWNER_APPROVED}")
print(f"host               : {ENV['host']}  (scratch={ENV['scratch']}, work={ENV['work']})")
print(f"python / platform  : {platform.python_version()} / {platform.platform()}")
print(f"torch              : {GPU['torch']}   cuda={GPU['cuda']}")
for g in GPU["gpus"]:
    print(f"  gpu              : {g['name']} {g['total_gib']} GiB {g['capability']} bf16={g['bf16']}")
if not N_GPU:
    print("  gpu              : none visible (expected for PREFLIGHT)")
print(f"dtype / TP         : {DTYPE} / {TENSOR_PARALLEL_SIZE}")
print(f"free scratch disk  : {free_gib} GiB")

WEIGHTS_GIB = 13.5
print()
print("resource estimate")
print(f"  weights at 16-bit            : {WEIGHTS_GIB} GiB download + load")
print(f"  per-config checkpoint copy   : ~{WEIGHTS_GIB} GiB written to scratch, then deleted (D8)")
print(f"  probe generation             : 200 prompts x <=1024 new tokens, once")
print(f"  arm (a) PILOT attack gens    : 200")
print(f"  arm (b) FULL  attack gens    : 125 configs x 200 = 25,000")
print(f"  utility evaluation           : top-10 configs x 3 datasets")
if DEVIATIONS:
    print()
    print("deviations recorded for this host:")
    for d in DEVIATIONS:
        print(f"  - {d}")

**Read this before raising `STAGE`.** Headroom after 13.5 GiB of 16-bit weights: a single 16 GiB
T4 or P100 leaves ~2.5 GiB for KV cache and activations, which is tight and needs a short
`max_model_len`; Kaggle's 2×T4 at tensor-parallel 2 leaves ~18.5 GiB; Colab L4 (24 GiB) leaves
~10.5 GiB; an 80 GiB A100/H100 leaves ~66 GiB.

Only Ampere or newer (A100, H100, L40S, L4) preserves the paper's bfloat16. If precision fidelity
matters for the claim, use such a host; if the goal is a free-tier check, accept and record the
float16 deviation.

## 2. Dependencies and the pinned repository

In [ ]:
def run(cmd, check=True):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-2000:])
    if r.returncode and check:
        print(r.stderr.strip()[-2000:])
        raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return r

# Released requirements.txt pins these.
REQUIRED = {"torch": "2.6.0", "transformers": "4.55.2", "datasets": "4.2.0", "vllm": "0.8.5"}

def check_deps() -> dict:
    import importlib.metadata as meta
    found = {}
    for pkg in REQUIRED:
        try:
            found[pkg] = meta.version(pkg)
        except meta.PackageNotFoundError:
            found[pkg] = None
    return found

DEPS = check_deps()
for pkg, want in REQUIRED.items():
    have = DEPS[pkg]
    mark = "ok" if have == want else ("MISSING" if have is None else "DIFFERENT")
    print(f"  {pkg:14s} want {want:9s} have {str(have):12s} {mark}")

if STAGE != "PREFLIGHT" and any(v is None for v in DEPS.values()):
    raise RuntimeError(
        "missing dependencies for a non-PREFLIGHT stage. Install the released pins first:\n"
        "  pip install -q torch==2.6.0 transformers==4.55.2 datasets==4.2.0 vllm==0.8.5\n"
        "On Colab/Kaggle the preinstalled torch may differ; a version mismatch is a deviation, "
        "not a silent substitution - record it in DEVIATIONS."
    )
for pkg, want in REQUIRED.items():
    if DEPS[pkg] not in (None, want):
        DEVIATIONS.append(f"{pkg}=={DEPS[pkg]} rather than the released pin {want}")

In [ ]:
# Clone the official repository at the pinned commit and verify we got exactly it.
REPO_DIR = ENV["work"] / "Steer2Edit"
if not REPO_DIR.exists():
    run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)])
run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_COMMIT])
head = run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"]).stdout.strip()
assert head == REPO_COMMIT, f"expected {REPO_COMMIT}, got {head}"
print("pinned commit verified:", head)

SAFETY = REPO_DIR / "safety_alignment"
sys.path.insert(0, str(SAFETY))
print("safety_alignment present:", SAFETY.is_dir())

## 3. Data validation

Checks the released artifacts before anything expensive. Three things matter: the probing sets are
the sizes the pipeline assumes (D1 — `mu` is the midpoint of two class means, which equals the
input-distribution mean only when the sets are balanced); the probe and evaluation data do not
overlap (D2 — the paper says probing uses AdvBench, the code uses MaliciousInstruct); and both
attacks are genuinely pre-released, so no attacker has to be trained.

In [ ]:
import csv as _csv

DATA = SAFETY / "data"
mal = [l.strip() for l in (DATA / "MaliciousInstruct.txt").read_text().splitlines() if l.strip()]
ben = [json.loads(l)["instruction"]
       for l in (DATA / "benign_instructions.jsonl").read_text().splitlines() if l.strip()]

with open(DATA / "advbench" / "harmful_behaviors.csv", newline="") as f:
    advbench = [r["goal"] for r in _csv.DictReader(f)]

gcg_dir = DATA / "gcg_results" / "llama2-7b"
gcg_files = sorted(gcg_dir.glob("adv_suffix_*.csv"))
gcg_rows = []
for p in gcg_files:
    with open(p, newline="") as f:
        gcg_rows += list(_csv.DictReader(f))

advllm_files = sorted((DATA / "advllm" / "advbench").rglob("*llama2*.pkl"))

print(f"probing  malicious (MaliciousInstruct) : {len(mal)}")
print(f"probing  benign    (Alpaca)            : {len(ben)}")
print(f"eval     AdvBench behaviours           : {len(advbench)}")
print(f"eval     GCG rows for llama2-7b        : {len(gcg_rows)} across {len(gcg_files)} files")
print(f"eval     ADV-LLM llama2 artifacts      : {[p.name for p in advllm_files]}")

assert len(mal) == len(ben) == 100, "D1/D4 assume 100 balanced probing pairs"
print()
print("balanced probing pairs (D1 midpoint estimator valid) : OK")

overlap = {m.strip().lower() for m in mal} & {a.strip().lower() for a in advbench}
print(f"probe/eval text overlap                             : {len(overlap)} items")
assert not overlap, "probing and evaluation data overlap - would invalidate the reproduction"

attacks_prereleased = bool(gcg_rows) and bool(advllm_files)
print(f"both attacks pre-released (no attacker training)     : {attacks_prereleased}")
assert attacks_prereleased

# The native endpoint, used unmodified. Read it out of the released utils.py source rather
# than importing the module: utils.py imports math_verify at top level (needed only for the
# utility scorer), which is not installed during a clean PREFLIGHT on Colab/Kaggle.
def load_refusal_signal(path):
    import ast
    tree = ast.parse(Path(path).read_text())
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(
            getattr(t, "id", None) == "REFUSAL_SIGNAL" for t in node.targets
        ):
            return ast.literal_eval(node.value)
    raise RuntimeError("REFUSAL_SIGNAL not found in released utils.py")

REFUSAL_SIGNAL = load_refusal_signal(SAFETY / "utils.py")
print()
print(f"native REFUSAL_SIGNAL keywords : {len(REFUSAL_SIGNAL)}")
print("  permissive examples:",
      [k for k in REFUSAL_SIGNAL if k in ("cannot", "I must", "I understand", "Hello!", "unethical")])
print("  -> this endpoint measures refusal-SHAPED TEXT, not withheld assistance.")

## 4. Edit-math validation (CPU)

Runs `tools/steer2edit_edit_math.py`, the reference transcription of the released arithmetic:
orientation, rank-1 structure, disabled-edit parity at `rho=-1`, edit scope, finite arithmetic,
exact restoration, determinism, zero-norm and zero-mean degenerate handling, bf16-vs-fp32 selection
agreement, and the structural result that MLP `|g|` cannot depend on `||mu||`.

In [ ]:
PROJECT_TOOLS = None
for cand in [Path.cwd() / "tools", Path.cwd().parent / "tools", ENV["work"] / "tools"]:
    if (cand / "steer2edit_edit_math.py").exists():
        PROJECT_TOOLS = cand
        break

if PROJECT_TOOLS is None:
    print("tools/steer2edit_edit_math.py not found next to this notebook.")
    print("Upload it (or clone the project repo) to run the edit-math suite here.")
else:
    sys.path.insert(0, str(PROJECT_TOOLS))
    import steer2edit_edit_math as s2e_math
    ok = s2e_math.run_validations()
    print()
    print("edit-math suite passed:", ok)
    assert ok, "edit-math validation failed - do not proceed to editing"

In [ ]:
# Soft-threshold arithmetic for the configuration under test. This is why the paper's
# Figure 6 reports near-zero MLP coefficients (section 5.2 of the deep study).
ra, rm, al = BEST_CFG["rho_attn"], BEST_CFG["rho_mlp"], BEST_CFG["alpha"]
print(f"gate |g| > rho*alpha    attn {ra * al:.3f}   mlp {rm * al:.3f}   "
      f"(mlp gate is {rm / ra:.2f}x stricter)")
print(f"gain 1/(rho(1-alpha))   attn {1 / (ra * (1 - al)):.2f}    mlp {1 / (rm * (1 - al)):.2f}")
print()
print("Any component with |g| below its gate receives no edit at all.")

## 5. Approval gate

Everything below downloads gated weights and generates text. It will not run until the owner
approves the scope in `docs/STEER2EDIT_REPRODUCTION_CONTRACT.md` and sets `OWNER_APPROVED = True`
with `STAGE` raised above `PREFLIGHT`.

In [ ]:
def require_approval(what: str) -> None:
    if STAGE == "PREFLIGHT":
        raise RuntimeError(
            f"{what} is not run during PREFLIGHT. Preflight is complete; this is the stopping "
            "point. Raise STAGE to SMOKE/PILOT/FULL and set OWNER_APPROVED=True after approval."
        )
    if not OWNER_APPROVED:
        raise RuntimeError(f"{what} requires OWNER_APPROVED=True (gated weights and GPU time).")

N_PROMPTS = {"PREFLIGHT": 0, "SMOKE": 8, "PILOT": 100, "FULL": 100}[STAGE]
GRID = {
    "SMOKE": [BEST_CFG],
    "PILOT": [BEST_CFG],
    "FULL": [
        {"rho_attn": round(a, 2), "rho_mlp": round(m, 2), "alpha": round(x, 2)}
        for a in [0.16, 0.18, 0.20, 0.22, 0.24]
        for m in [0.35, 0.40, 0.45, 0.50, 0.55]
        for x in [0.70, 0.75, 0.80, 0.85, 0.90]
    ],
}.get(STAGE, [])
print(f"stage {STAGE}: {N_PROMPTS} prompts, {len(GRID)} configuration(s)")

## 6. Calibration: probe responses and direction extraction

Two inherited choices are made explicit here rather than inherited silently. **D3**: the released
pipeline applies *no* refusal filtering — it keeps the first sampled response to every prompt. We
keep that behaviour for fidelity, but we *measure* how many positive-set responses actually look
like refusals, because that proportion is the assumption the whole direction rests on. **D6**: the
released sampler is unseeded, so the exact published directions are unrecoverable; we therefore
save the generated responses themselves as part of the run identity.

In [ ]:
require_approval("probe generation")

from vllm import LLM, SamplingParams
from transformers import AutoConfig, AutoTokenizer, GenerationConfig
import generate_response as _gr   # released module, for apply_chat parity

tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
gen_cfg = GenerationConfig.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
max_pos = AutoConfig.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION).max_position_embeddings

llm = LLM(model=BASE_MODEL, revision=BASE_MODEL_REVISION,
          tensor_parallel_size=TENSOR_PARALLEL_SIZE,
          max_model_len=min(1024 + 512, max_pos), dtype=DTYPE)

sp = SamplingParams(n=1, max_tokens=min(1024, max_pos),
                    temperature=float(gen_cfg.temperature),
                    top_p=float(gen_cfg.top_p), top_k=int(gen_cfg.top_k))
print("sampling params (released: copied from generation_config, unseeded):", sp)

mal_q, ben_q = mal[:N_PROMPTS], ben[:N_PROMPTS]
mal_out = llm.generate([_gr.apply_chat(q, tok) for q in mal_q], sp)
ben_out = llm.generate([_gr.apply_chat(q, tok) for q in ben_q], sp)

pos = [{"question": q, "response": o.outputs[0].text.strip()} for q, o in zip(mal_q, mal_out)]
neg = [{"question": q, "response": o.outputs[0].text.strip()} for q, o in zip(ben_q, ben_out)]

resp_dir = SAFETY / "responses"
resp_dir.mkdir(exist_ok=True)
(resp_dir / "llama2-7b_malicious.json").write_text(json.dumps(pos, indent=4))
(resp_dir / "llama2-7b_alpaca.json").write_text(json.dumps(neg, indent=4))

# Measure the D3 assumption instead of trusting it.
kw_refusal = sum(any(k in r["response"] for k in REFUSAL_SIGNAL) for r in pos)
print()
print(f"positive set: {kw_refusal}/{len(pos)} responses match a refusal keyword "
      f"({100 * kw_refusal / max(len(pos), 1):.1f}%)")
print("D3: the released pipeline does NOT filter for refusals; this fraction is the assumption "
      "the direction rests on. It is recorded in the run identity.")

In [ ]:
require_approval("direction extraction")

# Released extract_directions.py is a script, not a function. Run it as the authors do, so the
# token span (D5), the pre/post hook placement and the prompt-fixed pairing (D4) are theirs.
run([sys.executable, "extract_directions.py", "--model", "llama2-7b"])

import torch
DIRS_PATH = SAFETY / "directions" / "llama2-7b_refusal_dirs.pt"
dirs = torch.load(DIRS_PATH, map_location="cpu")
k0 = "model.layers[0].self_attn.o_proj"
print("direction keys      :", list(dirs.keys()))
print("post direction shape:", dirs["post"]["direction"][k0].shape)
print("pre  pos_mean shape :", dirs["pre"]["pos_mean"][k0].shape)

# Degenerate-direction instrumentation. Released code would silently produce "no edit" (5.1).
zero_dirs = [k for k, v in dirs["post"]["direction"].items() if float(v.norm()) == 0.0]
norms = torch.tensor([float(v.norm()) for v in dirs["post"]["direction"].values()])
print()
print(f"zero-norm directions: {len(zero_dirs)}  (released code would silently skip these)")
print(f"direction norms     : min {norms.min():.4f}  median {norms.median():.4f}  max {norms.max():.4f}")

## 7. Editing and on-model validation

The released `edit.py` asserts CUDA and writes a compact `(lambda, k_hat, v_hat)` pack. Before
trusting any edited model we re-check on the *real* weights what the CPU suite could only check on
a fixture: that a disabled edit is bitwise identical, that only the intended columns moved, that
each per-head change is rank 1, and that `save_pretrained` round-trips.

In [ ]:
require_approval("edit construction")

EDIT_ROOT = ENV["scratch"] / "edited_models"
EDIT_ROOT.mkdir(parents=True, exist_ok=True)

packs = []
for c in GRID:
    out_pt = EDIT_ROOT / "llama2-7b" / f"rank1__ra{c['rho_attn']}_rm{c['rho_mlp']}_a{c['alpha']}.pt"
    if not out_pt.exists():
        run([sys.executable, "edit.py", "--model", "llama2-7b",
             "--directions_path", str(DIRS_PATH),
             "--rho_attn", str(c["rho_attn"]), "--rho_mlp", str(c["rho_mlp"]),
             "--alpha", str(c["alpha"]), "--save_root", str(EDIT_ROOT)])
    packs.append((c, out_pt))

cfg0, pt0 = packs[0]
pack = torch.load(pt0, map_location="cpu")
n_attn, n_mlp = len(pack["attn"]), len(pack["mlp"])
print()
print(f"config {cfg0}: {n_attn} attention heads, {n_mlp} MLP neurons edited")
print("edited layers (attn):", sorted({l for l, _ in pack["attn"]}))
if n_mlp == 0:
    print("MLP set empty - consistent with Figure 6 and with the gate arithmetic in section 4.")

In [ ]:
require_approval("on-model edit validation")

from transformers import AutoModelForCausalLM
import evaluate as _ev   # released module: apply_rank1_edits_to_hf_model

DT = getattr(torch, DTYPE)
m = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION, torch_dtype=DT)
layers = m.model.layers

touched = sorted({l for l, _ in pack["attn"]})
before = {l: layers[l].self_attn.o_proj.weight.detach().clone() for l in touched}
control = next(l for l in range(len(layers)) if l not in touched)
control_before = layers[control].self_attn.o_proj.weight.detach().clone()

# (i) disabled-edit parity: an empty pack must change nothing.
_ev.apply_rank1_edits_to_hf_model(m, {"attn": {}, "mlp": {}, "meta": pack["meta"]})
assert torch.equal(layers[touched[0]].self_attn.o_proj.weight, before[touched[0]])
print("disabled-edit parity   : PASS (bitwise identical)")

# (ii) apply the real pack
_ev.apply_rank1_edits_to_hf_model(m, pack)

# (iii) scope: an unedited control layer must be untouched
assert torch.equal(layers[control].self_attn.o_proj.weight, control_before)
print(f"untouched control layer: PASS (layer {control})")

# (iv) rank-1 per edited head, using a RELATIVE singular-value test. An absolute tolerance
#      misreports the rank - see section 10 of the deep study.
H, A = m.config.hidden_size, m.config.num_attention_heads
D = H // A
ratios = []
for (l, h) in list(pack["attn"])[:8]:
    delta = (layers[l].self_attn.o_proj.weight[:, h * D:(h + 1) * D].float()
             - before[l][:, h * D:(h + 1) * D].float())
    sv = torch.linalg.svdvals(delta.double())
    ratios.append(float(sv[1] / sv[0]))
print(f"rank-1 per edited head : max s2/s1 = {max(ratios):.2e} over {len(ratios)} heads")
assert max(ratios) < 1e-3, "edited head delta is not rank 1"

# (v) finite arithmetic
assert torch.isfinite(layers[touched[0]].self_attn.o_proj.weight).all()
print("finite arithmetic      : PASS")

# (vi) save/reload fidelity, which the released pipeline relies on (D8)
EDITED_DIR = ENV["scratch"] / "edited_llama2_current"
m.save_pretrained(EDITED_DIR, safe_serialization=True)
tok.save_pretrained(EDITED_DIR)
m2 = AutoModelForCausalLM.from_pretrained(EDITED_DIR, torch_dtype=DT)
same = torch.equal(m2.model.layers[touched[0]].self_attn.o_proj.weight,
                   layers[touched[0]].self_attn.o_proj.weight)
print(f"save/reload fidelity   : {'PASS' if same else 'FAIL'}")
assert same
del m2

# (vii) restoration. The original checkpoint on disk is never mutated (D8); the in-memory model
#       is restored by subtracting the same factors.
neg_pack = {
    "attn": {k: {**v, "lambda": -v["lambda"]} for k, v in pack["attn"].items()},
    "mlp": {k: {"lambda": -v["lambda"]} for k, v in pack["mlp"].items()},
    "meta": pack["meta"],
}
_ev.apply_rank1_edits_to_hf_model(m, neg_pack)
resid = max(float((layers[l].self_attn.o_proj.weight.float() - before[l].float()).abs().max())
            for l in touched)
print(f"restoration residual   : {resid:.2e} ({DTYPE} rounding)")
del m

## 8. Attack generation and native evaluation

The endpoint is the released keyword list, unmodified. Reported per attack with its own
denominator, then averaged exactly as Table 5 does. We add an exact binomial interval, which the
paper does not report — labelled as our addition.

In [ ]:
require_approval("attack generation and evaluation")

def refusal_rate(texts):
    hits = [any(k in t for k in REFUSAL_SIGNAL) for t in texts]
    return sum(hits), len(hits)

def binom_ci(k, n, conf=0.95):
    """Exact Clopper-Pearson. OUR ADDITION: the paper reports no uncertainty."""
    from scipy.stats import beta
    if n == 0:
        return (float("nan"), float("nan"))
    lo = 0.0 if k == 0 else beta.ppf((1 - conf) / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - (1 - conf) / 2, k + 1, n - k)
    return (100 * lo, 100 * hi)

# Released evaluate.py drives generation for both attacks; it detects an edited_models/ path and
# materializes the checkpoint itself (D8). Run it per configuration and per attack, then parse
# ITS saved outputs - never hand-compute a headline number from stdout.
RESULTS = []
for c, pt in packs:
    for attack in ("gcg", "advllm"):
        run([sys.executable, "evaluate.py", "--model", str(pt), "--dataset", attack,
             "--tensor_parallel_size", str(TENSOR_PARALLEL_SIZE)])
        RESULTS.append({"config": c, "attack": attack, "pack": str(pt)})

print(f"completed {len(RESULTS)} (config, attack) evaluations")
print("Collect evaluate.py's saved result files into RESULTS before reporting any rate.")

## 9. Saved results, run identity and comparison

Nothing is a result until it is written with the identity that produced it. The identity pins the
checkpoint revision, the probe responses (unrecoverable otherwise, D6), the edit configuration, the
dtype, the library versions and every deviation.

In [ ]:
import datetime, hashlib

def sha256_16(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = ENV["work"] / f"steer2edit_native_{stamp}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

identity = {
    "created_utc": stamp,
    "stage": STAGE,
    "owner_approved": OWNER_APPROVED,
    "paper_version": "arXiv 2602.09870v4 (8 Nov 2026, NeurIPS 2026)",
    "repo_commit": REPO_COMMIT,
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "dtype": DTYPE,
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "host": ENV["host"],
    "gpus": GPU["gpus"],
    "library_versions": DEPS,
    "n_prompts": N_PROMPTS,
    "n_configs": len(GRID),
    "configs": GRID[:5] + (["..."] if len(GRID) > 5 else []),
    "probe_response_sha256_16": {
        "malicious": sha256_16(SAFETY / "responses" / "llama2-7b_malicious.json"),
        "alpaca": sha256_16(SAFETY / "responses" / "llama2-7b_alpaca.json"),
    },
    "directions_sha256_16": sha256_16(DIRS_PATH),
    "positive_set_keyword_refusal_fraction": kw_refusal / max(len(pos), 1),
    "native_endpoint": "released REFUSAL_SIGNAL keyword list, unmodified",
    "deviations": DEVIATIONS,
    "published_target": PUBLISHED,
}
(OUT_DIR / "run_identity.json").write_text(json.dumps(identity, indent=2))
(OUT_DIR / "results_raw.json").write_text(json.dumps(RESULTS, indent=2))
print("wrote", OUT_DIR / "run_identity.json")
print(json.dumps(identity, indent=2)[:1500])
print()
print("PRIVACY: raw prompts, model responses and per-case judgments stay in this run folder and "
      "must NOT be committed. The GitHub repository is public; only aggregates may leave.")

### How to read the comparison

The published target is average safety **88.15** (GCG 95.1,
ADV-LLM 81.2) at `rho_attn=0.18, rho_mlp=0.55,
alpha=0.8`. The contract calls our average safety reproduced at the configuration
level if it lands within ±5 percentage points, because the published value carries no interval and
the paper's `n` is undefined.

Three cautions when writing this up. The endpoint is keyword matching, so a high number means
refusal-*shaped* text, not withheld assistance — any claim about harmful assistance needs a
separate semantic measurement. Table 5's best-safety row also costs heavy utility
(avg 28.0), so it must not be quoted alongside the abstract's "matched
downstream performance" claim. And a reproduced number here says nothing about the multimodal
question: Appendix N evaluates a vision-language backbone on *text-only* benchmarks, so the
MM-SafetyBench four-condition evaluation remains a separate adaptation.

In [ ]:
# Final state report. A green preflight must never read as a completed reproduction.
print("=" * 78)
print(f"RUN STATE: {STAGE}   owner_approved={OWNER_APPROVED}")
print("=" * 78)
if STAGE == "PREFLIGHT":
    print("COMPLETED : platform detection, dependency check, pinned-commit verification,")
    print("            released-data validation, CPU edit-math suite, resource estimate.")
    print("NOT DONE  : no weights downloaded, no calibration, no edit, no generation,")
    print("            no evaluation, NO PUBLISHED NUMBER REPRODUCED.")
    print("NEXT      : owner approval for docs/STEER2EDIT_REPRODUCTION_CONTRACT.md, then")
    print("            STAGE='SMOKE' to prove the pipeline, then 'PILOT' for contract arm (a).")
else:
    print(f"prompts={N_PROMPTS}  configs={len(GRID)}  evaluations={len(RESULTS)}")
    print("Setup success is not a result. Confirm the saved results file is non-empty and that")
    print("its numbers come from evaluate.py's outputs, not from stdout.")
    if DEVIATIONS:
        print()
        print("deviations in this run:")
        for d in DEVIATIONS:
            print("  -", d)